# Survey Data Analysis — Reproducible Analysis

This notebook inventories the project's survey files and calculates detection metrics only after the researcher explicitly selects an approved dataset and maps its columns.

## Data-integrity rules
- No synthetic participants or example responses are generated.
- Do not commit identifiable or raw participant-level responses to a public repository.
- Warm-up records are excluded only when a phase/round column is explicitly mapped and the warm-up label is specified.
- Accuracy, precision, recall and F1 are calculated only when both verified ground-truth labels and participant predictions are available.
- Missing values, ambiguous labels and unknown schemas are reported rather than silently guessed.
- Files named `test_run`, `sample`, or `demo` are examples/test data and must not be used as study findings.

The previous notebook version generated six fictional rows (`P1`–`P6`). Those rows are demonstration data, not study findings. This version does not generate or analyze them as real participants.


In [ ]:
from pathlib import Path
import json
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

# Locate the repository root whether Jupyter starts in the root or notebooks/.
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data').is_dir() and ((candidate / 'src').is_dir() or (candidate / 'notebooks').is_dir()):
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Raw data directory exists:', RAW_DIR.exists())
print('Processed data directory exists:', PROCESSED_DIR.exists())

## 1. Inventory available data files

This section lists supported files and their dimensions. It deliberately does not print participant-level answers or personal identifiers.


In [ ]:
SUPPORTED_SUFFIXES = {'.csv', '.tsv', '.jsonl', '.json', '.xlsx', '.xls'}

def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == '.csv':
        # Try UTF-8 first; fall back to common spreadsheet export encoding.
        try:
            return pd.read_csv(path)
        except UnicodeDecodeError:
            return pd.read_csv(path, encoding='cp1251')
    if suffix == '.tsv':
        return pd.read_csv(path, sep='\t')
    if suffix == '.jsonl':
        return pd.read_json(path, lines=True)
    if suffix == '.json':
        return pd.read_json(path)
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    raise ValueError(f'Unsupported file type: {path.suffix}')

all_files = []
for folder in (RAW_DIR, PROCESSED_DIR):
    if folder.exists():
        all_files.extend(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in SUPPORTED_SUFFIXES)

inventory_rows = []
for path in sorted(set(all_files)):
    try:
        frame = read_table(path)
        relative_name = str(path.relative_to(PROJECT_ROOT))
        lower_name = path.stem.casefold()
        if any(marker in lower_name for marker in ('test_run', 'sample', 'demo', 'example')):
            file_role = 'TEST/DEMO — DO NOT USE FOR STUDY RESULTS'
        elif path.parent == RAW_DIR or RAW_DIR in path.parents:
            file_role = 'RAW SOURCE — check anonymization; do not publish as-is'
        else:
            file_role = 'Candidate processed dataset — verify before use'
        inventory_rows.append({
            'file': relative_name,
            'file_role': file_role,
            'rows': len(frame),
            'columns': len(frame.columns),
            'column_names': ', '.join(map(str, frame.columns)),
            'read_status': 'OK'
        })
    except Exception as exc:
        relative_name = str(path.relative_to(PROJECT_ROOT))
        lower_name = path.stem.casefold()
        if any(marker in lower_name for marker in ('test_run', 'sample', 'demo', 'example')):
            file_role = 'TEST/DEMO — DO NOT USE FOR STUDY RESULTS'
        elif path.parent == RAW_DIR or RAW_DIR in path.parents:
            file_role = 'RAW SOURCE — check anonymization; do not publish as-is'
        else:
            file_role = 'Candidate processed dataset — verify before use'
        inventory_rows.append({
            'file': relative_name,
            'file_role': file_role,
            'rows': None, 'columns': None, 'column_names': '',
            'read_status': f'ERROR: {type(exc).__name__}: {exc}'
        })

inventory = pd.DataFrame(inventory_rows)
if inventory.empty:
    print('No supported data files found in data/raw or data/processed.')
else:
    display(inventory)



## 2. Select the real analysis dataset

Set `DATA_FILE` to a project-relative path for the approved, anonymized response-level dataset. Leave it as `None` to stop before metric calculation. Aggregated summary tables are not substitutes for response-level data when calculating participant-level classification metrics.


In [ ]:
# Example only: set this to the actual file present in your project.
# Do not point this to a public copy of raw identifiable responses.
DATA_FILE = None  # e.g. 'data/processed/anonymized_responses_long.csv'

if DATA_FILE is None:
    df = None
    print('No dataset selected. Inventory is complete; metric calculations are intentionally skipped.')
else:
    selected_path = (PROJECT_ROOT / DATA_FILE).resolve()
    if PROJECT_ROOT not in selected_path.parents and selected_path != PROJECT_ROOT:
        raise ValueError('DATA_FILE must stay inside the project directory.')
    if not selected_path.is_file():
        raise FileNotFoundError(f'Dataset not found: {selected_path}')
    df = read_table(selected_path)
    print(f'Loaded {selected_path.relative_to(PROJECT_ROOT)}: {df.shape[0]} rows × {df.shape[1]} columns')
    print('Column names:', list(df.columns))
    print('Data types:')
    display(df.dtypes.astype(str).rename('dtype').to_frame())
    print('Missing values by column:')
    display(df.isna().sum().rename('missing_count').to_frame())
    print('Exact duplicate rows:', int(df.duplicated().sum()))


## 3. Map the dataset columns

Map columns using the exact names printed above. Do not infer mappings from column order. If survey responses are stored in wide format (one column per email), this notebook does not automatically reshape them; transform them to long format first and verify the transformation.


In [ ]:
# Fill these with exact column names from the selected dataset.
COLUMN_MAP = {
    'participant_id': None,
    'email_id': None,
    'true_label': None,       # ground-truth class for the email
    'predicted_label': None,  # participant's answer
    'confidence': None,       # optional confidence score
    'phase': None             # optional; must explicitly identify warm-up rows
}

# Set exact label values as they appear in the mapped columns.
POSITIVE_LABEL = 'phishing'
NEGATIVE_LABEL = 'legitimate'
WARMUP_LABEL = 'warmup'  # used only when COLUMN_MAP['phase'] is mapped

if df is not None:
    required_keys = ['true_label', 'predicted_label']
    missing_map = [key for key in required_keys if not COLUMN_MAP.get(key)]
    if missing_map:
        print('Metric calculation is not configured. Map these fields first:', missing_map)
        analysis_df = None
    else:
        absent = [col for col in COLUMN_MAP.values() if col and col not in df.columns]
        if absent:
            raise KeyError(f'Mapped columns not found in dataset: {absent}')
        analysis_df = df.copy()
        phase_col = COLUMN_MAP.get('phase')
        if phase_col:
            before = len(analysis_df)
            analysis_df = analysis_df.loc[analysis_df[phase_col].astype(str).str.strip().str.casefold() != WARMUP_LABEL.casefold()].copy()
            print(f'Excluded {before - len(analysis_df)} rows explicitly marked as warm-up.')
        else:
            print('No phase column mapped: no warm-up rows were excluded.')
        label_cols = [COLUMN_MAP['true_label'], COLUMN_MAP['predicted_label']]
        before = len(analysis_df)
        analysis_df = analysis_df.dropna(subset=label_cols).copy()
        print(f'Excluded {before - len(analysis_df)} rows with missing ground-truth/predicted labels.')
else:
    analysis_df = None


## 4. Classification metrics

Positive class is `phishing`; negative class is `legitimate` by default. Confirm these labels match the study ground truth before running. The participant answer is treated as the prediction; the independently verified email class is the ground truth.


In [ ]:
if analysis_df is not None:
    y_true = analysis_df[COLUMN_MAP['true_label']].astype(str).str.strip().str.casefold()
    y_pred = analysis_df[COLUMN_MAP['predicted_label']].astype(str).str.strip().str.casefold()
    pos = POSITIVE_LABEL.casefold()
    neg = NEGATIVE_LABEL.casefold()
    observed = set(y_true.unique()) | set(y_pred.unique())
    unexpected = observed - {pos, neg}
    if unexpected:
        raise ValueError(f'Unexpected label values: {sorted(unexpected)}. Normalize/map labels explicitly before analysis.')

    tp = int(((y_true == pos) & (y_pred == pos)).sum())
    tn = int(((y_true == neg) & (y_pred == neg)).sum())
    fp = int(((y_true == neg) & (y_pred == pos)).sum())
    fn = int(((y_true == pos) & (y_pred == neg)).sum())
    total = tp + tn + fp + fn
    if total == 0:
        raise ValueError('No valid classifications remain after filtering missing labels and warm-up rows.')

    safe_div = lambda a, b: (a / b) if b else np.nan
    metrics = pd.DataFrame([{
        'N_classifications': total,
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
        'Accuracy': safe_div(tp + tn, total),
        'Precision_phishing': safe_div(tp, tp + fp),
        'Recall_phishing': safe_div(tp, tp + fn),
        'Specificity_legitimate': safe_div(tn, tn + fp),
        'F1_phishing': safe_div(2 * tp, 2 * tp + fp + fn)
    }])
    display(metrics)

    confusion = pd.DataFrame([[tp, fn], [fp, tn]],
                             index=['Actual phishing', 'Actual legitimate'],
                             columns=['Predicted phishing', 'Predicted legitimate'])
    display(confusion)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(PROCESSED_DIR / 'classification_metrics.csv', index=False)
    confusion.to_csv(PROCESSED_DIR / 'classification_confusion_matrix.csv')
else:
    print('Metrics skipped: select a dataset and map true_label/predicted_label first.')


## 5. Confidence analysis (optional)

Confidence summaries are calculated only if a confidence column is mapped. Report the confidence scale used in the survey. The table reports count, mean, median and standard deviation; interpret small groups cautiously.


In [ ]:
if analysis_df is not None and COLUMN_MAP.get('confidence'):
    confidence_col = COLUMN_MAP['confidence']
    confidence_df = analysis_df.copy()
    confidence_df[confidence_col] = pd.to_numeric(confidence_df[confidence_col], errors='coerce')
    confidence_df = confidence_df.dropna(subset=[confidence_col])
    confidence_summary = confidence_df.groupby(COLUMN_MAP['predicted_label'])[confidence_col].agg(
        count='count', mean='mean', median='median', std='std'
    ).reset_index()
    display(confidence_summary)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    confidence_summary.to_csv(PROCESSED_DIR / 'confidence_summary.csv', index=False)

    ax = confidence_df.boxplot(column=confidence_col, by=COLUMN_MAP['predicted_label'], grid=True, figsize=(7, 4))
    ax.set_title('Confidence by participant response')
    ax.set_xlabel('Participant response')
    ax.set_ylabel('Confidence score')
    plt.suptitle('')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'survey_confidence_by_response.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('Confidence analysis skipped: no confidence column was mapped or no dataset was selected.')


## 6. Participant-level summary (optional)

This section produces an aggregate summary only. It does not export participant identifiers or row-level responses. It requires a participant ID column and reports the distribution of per-participant accuracy.


In [ ]:
if analysis_df is not None and COLUMN_MAP.get('participant_id'):
    pid_col = COLUMN_MAP['participant_id']
    participant_y_true = analysis_df[COLUMN_MAP['true_label']].astype(str).str.strip().str.casefold()
    participant_y_pred = analysis_df[COLUMN_MAP['predicted_label']].astype(str).str.strip().str.casefold()
    per_person = analysis_df.assign(
        _correct=(participant_y_true == participant_y_pred).astype(int)
    ).groupby(pid_col)['_correct'].agg(['count', 'mean']).rename(
        columns={'count': 'n_classifications', 'mean': 'accuracy'}
    )
    participant_summary = per_person['accuracy'].describe().to_frame('participant_accuracy').T
    participant_summary['participants'] = len(per_person)
    display(participant_summary)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    participant_summary.to_csv(PROCESSED_DIR / 'participant_accuracy_distribution.csv', index=False)
else:
    print('Participant summary skipped: participant_id is not mapped.')


## 7. Interpretation and reporting checklist

Before copying results into the paper, verify:
1. The selected data file is the approved, anonymized study dataset.
2. Ground-truth labels are verified independently of participant answers.
3. `phishing` is the positive class and `legitimate` is the negative class (or update the configuration).
4. Warm-up rows are excluded only using a documented phase marker and the exclusion rule is reported.
5. The unit of analysis is clear: email classifications are not the same as unique participants.
6. Metrics are reported with denominators; avoid claiming generalizability from a small or convenience sample.
7. Any automated-system comparison is omitted unless actual baseline predictions and a fair comparison protocol exist.

**Current limitation:** this notebook does not infer the survey schema or fabricate findings. Results appear only after `DATA_FILE` and `COLUMN_MAP` are configured with the real study data.
